# Qwen-Image Generator on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hirannalaka19/omnivoice-colab/blob/main/Qwen_Image_Colab.ipynb)

Generate images with **[Qwen-Image](https://huggingface.co/Qwen/Qwen-Image)** - Alibaba's 20B
text-to-image model, Apache-2.0 - in a Gradio UI, on a Colab **A100** GPU.

It is the current best open model for **text inside images**, in English and Chinese: signs,
posters, slides, whole paragraphs.

---

## Before you start (one time, 1 minute)

1. **Pick the GPU** - `Runtime` -> `Change runtime type` -> **A100 GPU** -> *Save*.
   A100 needs Colab Pro. L4 and T4 also work; the app drops to 4-bit automatically.
2. That is genuinely it. Unlike FLUX.1-dev, Qwen-Image is **ungated** - no licence to accept
   and no token needed.

> Optional: a Hugging Face **Read** token in Colab Secrets (key icon in the left sidebar ->
> *Add new secret* -> name `HF_TOKEN`, enable *Notebook access*) lifts the throttle the Hub
> puts on anonymous downloads. Worth it, since the weights are 57 GB.

## Then just run the cells in order

`1` check GPU -> `2` install -> `3` download weights -> `4` run the app, and click the
`https://....gradio.live` link that appears.

> **In a hurry?** In step 3 pick `diffusers/qwen-image-nf4` - the same model pre-quantised,
> so it downloads 28 GB instead of 57 GB and still runs on any GPU Colab hands you.

In [ ]:
#@title 1. Check the GPU { display-mode: "form" }
#@markdown Runtime -> Change runtime type -> **A100 GPU**, then run this cell.

!nvidia-smi

import shutil

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU attached. Runtime -> Change runtime type -> GPU (A100 recommended), then re-run."
    )

name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_disk = shutil.disk_usage("/content").free / 1024**3

print(f"\nGPU:        {name}  -  {vram:.0f} GB VRAM")
print(f"Free disk:  {free_disk:.0f} GB  (Qwen-Image is ~57 GB, the NF4 build ~28 GB)")
print(f"torch:      {torch.__version__}")

# The bf16 transformer on its own is 38 GiB, so only an 80 GB card runs it resident.
if vram >= 62:
    print("\nMode: bf16, whole pipeline on the GPU  ->  ~10 s per 8-step image.")
elif vram >= 42:
    print("\nMode: bf16 + CPU offload  ->  full quality, a little slower.")
elif vram >= 20:
    print("\nMode: 4-bit NF4  ->  ~15-25 s per 8-step image on an A100 40GB.")
    print("      bf16 does not fit on this card: the transformer needs 38 GiB before activations.")
else:
    print("\nMode: 4-bit NF4  ->  it works, but expect minutes per image. A100 is much nicer.")

if free_disk < 65:
    print(
        f"\nWARNING: only {free_disk:.0f} GB free. Choose 'diffusers/qwen-image-nf4' in step 3"
        " (~28 GB), or the download may fail part way through."
    )

In [ ]:
#@title 2. Install Qwen-Image + the app { display-mode: "form" }
#@markdown Pulls diffusers, gradio and friends. Colab's PyTorch is left alone on purpose. Takes ~2 minutes.

%cd /content/
!rm -rf ./omnivoice-colab
!git clone https://github.com/hirannalaka19/omnivoice-colab.git
%cd ./omnivoice-colab
!pip install -r qwen_colab.txt

from IPython.display import clear_output

clear_output()

import importlib

missing = []
for module in ("diffusers", "transformers", "accelerate", "gradio", "peft", "bitsandbytes", "huggingface_hub"):
    try:
        version = importlib.import_module(module).__version__
        print(f"{module:<18} {version}")
    except Exception as err:
        missing.append(f"{module} ({err})")

# The whole notebook hangs off this one class, so check it directly.
try:
    from diffusers import QwenImagePipeline  # noqa: F401

    print(f"{'QwenImagePipeline':<18} available")
except Exception as err:
    missing.append(f"QwenImagePipeline ({err})")

if missing:
    print("\nSomething did not install:", ", ".join(missing))
    print("Re-run this cell. If it keeps failing: Runtime -> Restart session, then run it again.")
else:
    print("\nInstall complete. Run cell 3 next.")

In [ ]:
#@title 3. Download the model weights { display-mode: "form" }
#@markdown One time per session. Qwen-Image is **ungated** and Apache-2.0 - there is no licence
#@markdown to accept and no token to create. An `HF_TOKEN` in Colab Secrets is optional: it only
#@markdown lifts the throttle the Hub puts on anonymous downloads.
#@markdown
#@markdown Short on time or disk? Pick `diffusers/qwen-image-nf4` - the same model pre-quantised,
#@markdown 28 GB instead of 57 GB.

model_id = "Qwen/Qwen-Image" #@param ["Qwen/Qwen-Image", "Qwen/Qwen-Image-2512", "diffusers/qwen-image-nf4"] {allow-input: true}
lightning = "8-step (Fast)" #@param ["8-step (Fast)", "4-step (Turbo)", "none - 50-step base model only"]

import os
import subprocess
import sys
import time

sys.path.insert(0, "/content/omnivoice-colab")
from qwen_image_app import lightning_for  # keeps the LoRA filenames in one place

token = ""
try:
    from google.colab import userdata

    token = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    token = ""

if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN found in Colab Secrets - downloads will not be throttled.")
else:
    print("No HF_TOKEN - fine, Qwen-Image is ungated. Add one if the download crawls.")

# Only the diffusers-format folders, in case the repo ever grows a single-file checkpoint.
DIFFUSERS_ONLY = [
    "model_index.json",
    "scheduler/*",
    "tokenizer/*",
    "text_encoder/*",
    "transformer/*",
    "vae/*",
]


def snapshot(repo, patterns, disable_xet):
    # Run in a clean subprocess: huggingface_hub reads the Xet switch at import time.
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    code = (
        "from huggingface_hub import snapshot_download\n"
        "snapshot_download(%r, allow_patterns=%r, max_workers=8)\n" % (repo, patterns)
    )
    return subprocess.call([sys.executable, "-c", code], env=env) == 0


def pull(repo, patterns, label):
    # Hugging Face's CDN occasionally rejects its own signed URLs. Finished files are
    # cached, so a retry only fetches what is still missing.
    for attempt in range(1, 4):
        print(f"\nDownloading {label} - plain HTTP (attempt {attempt}/3) ...")
        if snapshot(repo, patterns, disable_xet=True):
            return True
        print("Hit a CDN error - retrying, already-downloaded shards are kept ...")
        time.sleep(5)
    print("\nFalling back to the Xet transfer path (can be slower on Colab) ...")
    return snapshot(repo, patterns, disable_xet=False)


if not pull(model_id, DIFFUSERS_ONLY, model_id):
    raise SystemExit(
        "Download failed. Common causes: the disk filled up, the repo id is misspelt, or "
        "huggingface.co is having an incident (https://status.huggingface.co). Fix and "
        "re-run - nothing is downloaded twice."
    )

# The Lightning LoRA is what turns 50 steps into 8, so it is worth pre-fetching too.
wanted = {"8-step (Fast)": 8, "4-step (Turbo)": 4}.get(lightning)
if wanted:
    lora_repo, files = lightning_for(model_id)
    weight_name = files[wanted]
    if not pull(lora_repo, [weight_name], f"{weight_name} (~0.9 GB)"):
        print(f"\nCould not fetch {weight_name}. The app still works - set Speed to 'Quality'.")

print(f"\n{model_id} is cached and ready. Run cell 4.")

In [ ]:
#@title 4. Run the Qwen-Image generator { display-mode: "form" }
#@markdown Wait for the **`https://....gradio.live`** link, open it, and start generating.
#@markdown Leave this cell running - stopping it closes the app.

import os

try:
    from google.colab import userdata

    secret = (userdata.get("HF_TOKEN") or "").strip()
    if secret:
        os.environ["HF_TOKEN"] = secret
except Exception:
    pass

%cd /content/omnivoice-colab
!python qwen_image_app.py

In [ ]:
#@title Optional: generate without the UI { display-mode: "form" }
#@markdown Prefer plain cells to a web UI? Stop cell 4 first (the app holds the GPU), then run this.

prompt = "a bookshop window at dusk, a hand-lettered chalkboard sign reading 'OPEN LATE - POETRY UPSTAIRS', warm lamplight, rain on the glass, 35mm photograph" #@param {type:"string"}
negative_prompt = " " #@param {type:"string"}
model_id = "Qwen/Qwen-Image" #@param ["Qwen/Qwen-Image", "Qwen/Qwen-Image-2512", "diffusers/qwen-image-nf4"] {allow-input: true}
size = "1:1  1328 x 1328" #@param ["1:1  1328 x 1328", "16:9  1664 x 928", "9:16  928 x 1664", "4:3  1472 x 1104", "3:4  1104 x 1472", "3:2  1584 x 1056", "2:3  1056 x 1584", "1:1  1024 x 1024"]
speed = "8 steps (Lightning)" #@param ["8 steps (Lightning)", "4 steps (Lightning)", "50 steps (base model)"]
seed = -1 #@param {type:"integer"}
num_images = 1 #@param {type:"slider", min:1, max:4, step:1}

import os
import random
import sys
import time

import torch
from IPython.display import display

sys.path.insert(0, "/content/omnivoice-colab")

try:
    from google.colab import userdata

    os.environ["HF_TOKEN"] = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    pass

# Reuse the app's loader, so precision and the Lightning scheduler are handled the same way.
from qwen_image_app import RUNNER, OUTPUT_DIR, decorate, save_image

SIZES = {
    "1:1  1328 x 1328": (1328, 1328),
    "16:9  1664 x 928": (1664, 928),
    "9:16  928 x 1664": (928, 1664),
    "4:3  1472 x 1104": (1472, 1104),
    "3:4  1104 x 1472": (1104, 1472),
    "3:2  1584 x 1056": (1584, 1056),
    "2:3  1056 x 1584": (1056, 1584),
    "1:1  1024 x 1024": (1024, 1024),
}

want = {"8 steps (Lightning)": 8, "4 steps (Lightning)": 4}.get(speed)
steps = want or 50
# Lightning is distilled without classifier-free guidance, so CFG stays at 1.
cfg = 1.0 if want else 4.0
width, height = SIZES[size]

mode = RUNNER.load(model_id, "auto")
RUNNER.set_speed(want)
print(f"{model_id} loaded ({mode}) - {steps} steps, true CFG {cfg}, {width}x{height}\n")

full_prompt = decorate(prompt, True)
for i in range(int(num_images)):
    this_seed = random.randint(0, 2**31 - 1) if int(seed) < 0 else int(seed) + i
    started = time.time()
    image = RUNNER.pipe(
        prompt=full_prompt,
        negative_prompt=negative_prompt or " ",
        width=width,
        height=height,
        num_inference_steps=steps,
        true_cfg_scale=cfg,
        generator=torch.Generator("cpu").manual_seed(this_seed),
    ).images[0]
    path = save_image(
        image, full_prompt, negative_prompt, this_seed, steps, cfg, model_id, (width, height), want
    )
    print(f"seed {this_seed} - {time.time() - started:.1f}s - saved to {path}")
    display(image)

In [ ]:
#@title Optional: copy the generated images to Google Drive { display-mode: "form" }
#@markdown Colab wipes `/content` when the runtime disconnects. This copies everything to
#@markdown `MyDrive/Qwen_Output` so it survives.

import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

source = Path("/content/Qwen_Output")
target = Path("/content/drive/MyDrive/Qwen_Output")
target.mkdir(parents=True, exist_ok=True)

copied = 0
for item in sorted(source.glob("*")):
    if item.is_file():
        shutil.copy2(item, target / item.name)
        copied += 1

print(f"Copied {copied} file(s) to {target}")

---

## Credits & licence

* Model: **[Qwen/Qwen-Image](https://huggingface.co/Qwen/Qwen-Image)** by the Qwen team, Alibaba
* Few-step LoRAs: **[lightx2v/Qwen-Image-Lightning](https://huggingface.co/lightx2v/Qwen-Image-Lightning)**
  by [ModelTC](https://github.com/ModelTC/Qwen-Image-Lightning)
* Inference: [Hugging Face diffusers](https://github.com/huggingface/diffusers)
* Colab wrapper & Gradio app by [HiranNalaka](https://github.com/hirannalaka19)

**Qwen-Image is Apache-2.0**, and so are the Lightning LoRAs - commercial use is allowed, and
no token or licence click is required.

## Usage disclaimer

You are responsible for what you generate. Do not use this to create sexual content involving
minors, non-consensual intimate imagery, deepfakes of real people intended to deceive,
harassment, disinformation, or anything else that is illegal where you live.